# PersonaQA Results

Computes accuracy for three model variants across three methods:
- **Zero-shot**
- **LIT** (averaged across layers 1–15)
- **Patchscopes** (any-correct across target layers, averaged across source layers 1–15)

In [ ]:
import json
import os
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

TASKS = [
    "country",
    "favorite_food",
    "favorite_drink",
    "favorite_music_genre",
    "favorite_sport",
    "favorite_boardgame",
]

VARIANTS = ["PersonaQA", "PersonaQA-Fantasy", "PersonaQA-Shuffled"]

BASE_DIR = Path("/home/li.mil/verb_faithfulness/results/personaqa")  # Change this

## Helper Functions

In [ ]:
def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with open(path) as f:
        return json.load(f)

def check_correct(completion, answer):
    if not completion or not answer:
        return False
    return answer.lower() in completion.lower()

def compute_accuracy(results):
    if not results:
        return None
    correct = sum(1 for item in results.values() if check_correct(item.get("completion"), item.get("answer")))
    return correct / len(results)

def get_patchscopes_dirs(variant_dir):
    """Return list of (src, tgt, path) tuples from patchscopes output dirs."""
    ps_dir = variant_dir / "patchscopes"
    if not ps_dir.exists():
        return []
    configs = []
    for d in ps_dir.iterdir():
        if d.is_dir() and "src=" in d.name and "tgt=" in d.name:
            parts = d.name.split("_")
            src = int(parts[0].split("=")[1])
            tgt = int(parts[1].split("=")[1])
            configs.append((src, tgt, d))
    return sorted(configs)

def get_lit_layers(variant_dir):
    lit_dir = variant_dir / "lit"
    if not lit_dir.exists():
        return []
    return sorted([int(d.name) for d in lit_dir.iterdir() if d.is_dir() and d.name.isdigit()])

## Load and Compute Accuracy Per Variant

In [ ]:
all_results = {}  # variant -> method -> task -> accuracy

for variant in VARIANTS:
    vdir = BASE_DIR / variant
    all_results[variant] = {}

    # Zero-shot
    zs = {}
    for task in TASKS:
        zs[task] = compute_accuracy(load_json(vdir / "zeroshot" / f"{task}.json"))
    all_results[variant]["Zero-shot"] = zs

    # LIT — average across layers
    layers = get_lit_layers(vdir)
    lit = {}
    for task in TASKS:
        accs = [compute_accuracy(load_json(vdir / "lit" / str(l) / f"{task}.json")) for l in layers]
        accs = [a for a in accs if a is not None]
        lit[task] = np.mean(accs) if accs else None
    all_results[variant]["LIT"] = lit

    # Patchscopes — any-correct across tgt layers, averaged across src layers
    configs = get_patchscopes_dirs(vdir)
    src_layers = sorted(set(src for src, _, _ in configs))
    ps = {}
    for task in TASKS:
        src_accs = []
        for src in src_layers:
            tgt_dirs = [(tgt, d) for s, tgt, d in configs if s == src]
            if not tgt_dirs:
                continue
            first_results = load_json(tgt_dirs[0][1] / f"{task}.json")
            if first_results is None:
                continue
            n = len(first_results)
            correct = 0
            for idx in range(n):
                any_correct = False
                for tgt, d in tgt_dirs:
                    r = load_json(d / f"{task}.json")
                    if r and str(idx) in r and check_correct(r[str(idx)].get("completion"), r[str(idx)].get("answer")):
                        any_correct = True
                        break
                if any_correct:
                    correct += 1
            src_accs.append(correct / n)
        ps[task] = np.mean(src_accs) if src_accs else None
    all_results[variant]["Patchscopes"] = ps

print("Loaded results for:", list(all_results.keys()))

## Accuracy Per Variant

In [ ]:
for variant in VARIANTS:
    print(f"\n{'='*70}")
    print(f"  {variant}")
    print(f"{'='*70}")
    rows = {}
    for method in ["Zero-shot", "LIT", "Patchscopes"]:
        rows[method] = all_results[variant][method]
    df = pd.DataFrame(rows).T
    df["AVERAGE"] = df.mean(axis=1)
    print(df.round(4).to_string())

## Summary: Average Accuracy Across Tasks, By Variant and Method

In [ ]:
summary = {}
for variant in VARIANTS:
    summary[variant] = {}
    for method in ["Zero-shot", "LIT", "Patchscopes"]:
        accs = [v for v in all_results[variant][method].values() if v is not None]
        summary[variant][method] = np.mean(accs) if accs else None

df_summary = pd.DataFrame(summary).T
print("\nAVERAGE ACCURACY ACROSS TASKS")
print("="*60)
print(df_summary.round(4).to_string())